# Objective: Build a machine learning model that classifies the sentiment of text data (positive, negative, or neutral), providing insights into public opinion or customer feedback. 

### Tech Stack: Python, pandas, scikit-learn, NLTK or TextBlob, matplotlib/seaborn, Jupyter Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string


In [ ]:
df=pd.read_csv("data\Reviews.csv")

In [ ]:
df.head()

In [ ]:
#number of rows and columns
df.shape

In [ ]:
#columns
df.columns

In [ ]:
#refining column names
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(" ","_")
df.columns

In [ ]:
#watching null values
df.isnull().any()

In [ ]:
#number of null values
df.isnull().sum()

The dataset contains very few missing values. Only the ProfileName (26) and Summary (27) columns have missing entries. Since the sentiment analysis model will use only the Text column as input and the Score column as the target variable, these missing values do not affect the analysis. Therefore, no rows were removed due to missing values.

In [ ]:
df["score"].value_counts().sort_index()

Nearly 64% of all reviews are 5-star reviews.
That means Amazon customers are generally more likely to leave positive reviews than negative ones.
Therefore, this is an imbalanced dataset

In [ ]:
#checking for duplicate rows
df.duplicated().any()

In [ ]:
#creating a new column called sentiments
def get_sentiment(score):
    if score <=2:
        return 'Negative'
    elif score ==3:
        return 'Neutral'
    else:
        return 'Positive'

df["sentiment"] = df["score"].apply(get_sentiment)

In [ ]:
#checking
df[['score', 'sentiment']].head(10)

In [ ]:
df['sentiment'].value_counts()

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='sentiment', palette='Set2')
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.tight_layout()
plt.savefig('images/sentiment_distribution.png')
plt.show()

The dataset is highly imbalanced, with approximately 78% positive, 14% negative, and 8% neutral reviews. This suggests that most customers tend to leave favorable reviews. While this reflects real customer behavior, the imbalance may cause machine learning models to favor the majority (positive) class. Therefore, evaluation metrics such as precision, recall, and F1-score will be considered alongside accuracy.

### Text processing 

In [ ]:
!pip install nltk

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
#doing for one review at a time
sample_review = df.loc[0, "text"]

print(sample_review)

In [ ]:
#lowercasing
lower_review = sample_review.lower()
print("lower cased review:")
print(lower_review)

#punctuation removal
no_punct = lower_review.translate(str.maketrans('', '', string.punctuation))
print("\nreview with punctuations removed: ")
print(no_punct)

#tokeinization
tokens = word_tokenize(no_punct)
print("\ntokenised review: ")
print(tokens)

#stopword removal
stop_words = set(stopwords.words('english'))

#filtering
filtered_tokens = [
    word for word in tokens
    if word not in stop_words
]
print("\nTokens after filtering: ")
print(filtered_tokens)

- All review text is converted to lowercase to ensure that words such as "Good", "GOOD", and "good" are treated as the same token, reducing the vocabulary size.
- Punctuation marks generally do not contribute to sentiment in a basic text classification task. Removing them simplifies the text and avoids treating words like "great!" and "great" as different tokens.
- Tokenization is the process of splitting a sentence into individual words (tokens). This enables further preprocessing operations such as stopword removal and lemmatization.
- Stopwords are frequently occurring words (e.g., "the", "is", "and") that usually carry little semantic meaning. Removing them reduces noise and helps the model focus on informative words.
- Lemmatization converts words to their base (dictionary) form, such as "cars" to "car" or "studies" to "study". This reduces vocabulary size while preserving the actual meaning of words.

To avoid repeating the same preprocessing steps for every review, a reusable function is created. This function applies all preprocessing operations sequentially, forming a text preprocessing pipeline.

### Text processing pipeline

In [ ]:
#now for the complete dataset
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Tokenize
    tokens = word_tokenize(text)

    # Keep only alphabetic words
    tokens = [word for word in tokens if word.isalpha()]

    # Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # Lemmatize
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    # Join words back into a sentence
    text = " ".join(tokens)

    return text

Before training a machine learning model, textual data must be cleaned and standardized. Text preprocessing reduces noise by converting text into a consistent format, allowing the model to focus on meaningful words rather than differences in capitalization, punctuation, or common stopwords.

In [ ]:
#testing the above defined text processing pipeline on one of the review
sample_review = df.loc[0, "text"]

print("Original Review:\n")
print(sample_review)

print("\n" + "="*80 + "\n")

print("Cleaned Review:\n")
print(preprocess_text(sample_review))

In [ ]:
#applying the function to all the 568,454 reviews
df["clean_text"] = df["text"].apply(preprocess_text)

This preprocessing step takes around seven minutes on the full Amazon review dataset because each review undergoes tokenization, stopword removal, and lemmatization. Since it's a one-time preprocessing step, the cleaned dataset can be saved and reused in subsequent runs.

In [ ]:
df[['text','clean_text']].head()

### TF-IDF Vectorization

Machine learning algorithms cannot process raw text directly. TF-IDF (Term Frequency–Inverse Document Frequency) converts textual data into numerical feature vectors by assigning higher weights to words that are frequent in a document but rare across the dataset. This helps the model focus on informative words while reducing the influence of commonly occurring words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["clean_text"])

In [ ]:
feature_names = vectorizer.get_feature_names_out()
print(feature_names[:20])

In [ ]:
print(type(X))

print(X.shape)

print(X)

### preparing the ML model

In [ ]:
from sklearn.model_selection import train_test_split

X = vectorizer.fit_transform(df["clean_text"])
y = df["sentiment"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

The dataset is divided into training (80%) and testing (20%) subsets. The training set is used to learn patterns from the data, while the testing set evaluates the model on unseen reviews. Stratified sampling is applied to preserve the original class distribution across both subsets

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
print(y_train.value_counts())

print(y_test.value_counts())

### Implementing Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_model = MultinomialNB()
vectorizer = TfidfVectorizer()
nb_model.fit(X_train, y_train)

In [ ]:
y_pred_nb = nb_model.predict(X_test)
print(y_pred_nb[:10])

### Accuracy prediction

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", accuracy)
print(f"Accuracy: {accuracy:.2%}")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred_nb))

Although the Naive Bayes classifier achieved an overall accuracy of 79.2%, the classification report revealed that it performed well only on the majority (Positive) class. The model struggled to correctly identify Neutral and Negative reviews, indicating that accuracy alone is not a sufficient metric for evaluating performance on an imbalanced dataset.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_nb)
print(cm)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred_nb)

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Reds',
    xticklabels=['Negative','Neutral','Positive'],
    yticklabels=['Negative','Neutral','Positive']
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - Naive Bayes")
plt.savefig('images/heatmap_nb.png')
plt.show()

The confusion matrix provides a detailed summary of the model's predictions by comparing actual and predicted sentiment classes. The diagonal values represent correct predictions, while off-diagonal values indicate misclassifications. It helps identify which classes the model struggles to distinguish.

### Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model = LogisticRegression(max_iter=1000,random_state=42)
lr_model.fit(X_train, y_train)
y_pred_lr=lr_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"Accuracy: {accuracy_lr:.2%}")
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.metrics import confusion_matrix

cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm_lr,
    annot=True,
    fmt='d',
    cmap='Greens',
    xticklabels=['Negative','Neutral','Positive'],
    yticklabels=['Negative','Neutral','Positive']
)

plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")
plt.title("Confusion Matrix - Logistic Regression")
plt.savefig('images/heatmap_lr.png')
plt.show()

Logistic Regression is trained using the TF-IDF feature vectors to classify reviews into Positive, Negative, and Neutral sentiments. Unlike Naive Bayes, Logistic Regression learns weights for each feature, enabling it to model the relationship between words and sentiment classes more effectively.

Two machine learning models were trained for sentiment classification: Multinomial Naive Bayes and Logistic Regression. While Naive Bayes provided a simple baseline model, Logistic Regression achieved higher accuracy and significantly better Precision, Recall, and F1-scores across all sentiment classes. Therefore, Logistic Regression was selected as the better-performing model.

### WordCloud

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

In [ ]:
def plot_wordcloud(sentiment):

    text = " ".join(df[df["sentiment"] == sentiment]["clean_text"])

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color="white"
    ).generate(text)

    plt.figure(figsize=(10,5))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"WordCloud - {sentiment}")
    plt.savefig(f'images/wordcloud{sentiment}.png')
    plt.show()

In [ ]:
plot_wordcloud('Negative')

In [ ]:
plot_wordcloud('Positive')

In [ ]:
plot_wordcloud('Neutral')

WordClouds are generated for each sentiment class to visualize the most frequently occurring words. Larger words indicate higher frequency and provide an intuitive understanding of the vocabulary associated with Positive, Negative, and Neutral reviews.

### Inspecting misclassified reviews

In [ ]:
misclassified = df.iloc[y_test.index].copy()

misclassified["Actual"] = y_test.values
misclassified["Predicted"] = y_pred_lr

errors = misclassified[
    misclassified["Actual"] != misclassified["Predicted"]
]
errors[["text","Actual","Predicted"]].head(5)

In [ ]:
models = ['Naive Bayes', 'Logistic Regression']
accuracies = [accuracy, accuracy_lr]

plt.figure(figsize=(6,4))

sns.barplot(x=models, y=accuracies, palette='Set2')

plt.ylim(0,1)
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")

for i, value in enumerate(accuracies):
    plt.text(i, value+0.01, f"{value:.2%}", ha='center', fontsize=11)
plt.savefig('images/model_acc_comparison.png')
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Macro averages
nb_precision, nb_recall, nb_f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred_nb,
    average='macro'
)

lr_precision, lr_recall, lr_f1, _ = precision_recall_fscore_support(
    y_test,
    y_pred_lr,
    average='macro'
)

In [ ]:
comparison = pd.DataFrame({
    'Metric':['Precision','Recall','F1 Score'],
    'Naive Bayes':[nb_precision, nb_recall, nb_f1],
    'Logistic Regression':[lr_precision, lr_recall, lr_f1]
})

comparison

In [ ]:
comparison.set_index('Metric').plot(
    kind='bar',
    figsize=(8,5)
)

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0,1)
plt.legend(loc='upper right')
plt.savefig('images/model_perf_compaarison.png')
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1,2, figsize=(12,5))

ConfusionMatrixDisplay(cm).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title("Naive Bayes")

ConfusionMatrixDisplay(cm_lr).plot(ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title("Logistic Regression")

plt.tight_layout()
plt.savefig('images/confusion_matrices.png')
plt.show()

In [ ]:
nb_report = classification_report(
    y_test,
    y_pred_nb,
    output_dict=True
)

lr_report = classification_report(
    y_test,
    y_pred_lr,
    output_dict=True
)
classes = ['Negative','Neutral','Positive']

nb_f1 = [nb_report[c]['f1-score'] for c in classes]
lr_f1 = [lr_report[c]['f1-score'] for c in classes]

In [ ]:
x = np.arange(len(classes))
width = 0.35

plt.figure(figsize=(8,5))

plt.bar(x-width/2, nb_f1, width, label='Naive Bayes')
plt.bar(x+width/2, lr_f1, width, label='Logistic Regression')

plt.xticks(x, classes)
plt.ylim(0,1)

plt.ylabel("F1 Score")
plt.title("Class-wise F1 Score Comparison")
plt.legend()
plt.savefig('images/f1_score_comp.png')
plt.show()

# Conclusion

A complete sentiment analysis pipeline was developed using Amazon product reviews. The text data was preprocessed through lowercasing, punctuation removal, tokenization, stopword removal, and lemmatization before being transformed into numerical features using TF-IDF vectorization. Two machine learning models, Multinomial Naive Bayes and Logistic Regression, were trained and evaluated. Logistic Regression achieved the best overall performance with an accuracy of 87.69%, outperforming Naive Bayes in Precision, Recall, and F1-score across all sentiment classes. The project demonstrates how Natural Language Processing (NLP) and Machine Learning can automatically analyze customer opinions and can be applied to product review analysis, customer feedback monitoring, brand reputation management, and social media sentiment analysis.